In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os, random
import cv2
import numpy as np
from tqdm import tqdm


OUT_DIR = "/kaggle/working/frames"      
MAX_PER_CLASS = 300                     
N_FRAMES = 30                           
FACE_SIZE = 224
VIDEO_EXT = (".mp4", ".avi", ".mov", ".mkv", ".webm", ".MP4")
random.seed(42)

detector = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
assert not detector.empty(), "Haar cascade failed to load - reinstall opencv-python"

def detect_face(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = detector.detectMultiScale(gray, 1.1, 5, minSize=(60, 60))
    return max(faces, key=lambda b: b[2] * b[3]) if len(faces) else None

def crop_face(frame, box, margin=0.2):
    x, y, w, h = box
    mx, my = int(w * margin), int(h * margin)
    x1, y1 = max(x - mx, 0), max(y - my, 0)
    x2, y2 = min(x + w + mx, frame.shape[1]), min(y + h + my, frame.shape[0])
    return cv2.resize(frame[y1:y2, x1:x2], (FACE_SIZE, FACE_SIZE))

def extract_faces(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step = max(total // (N_FRAMES * 2), 1)       
    faces, idx = [], 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if idx % step == 0:
            box = detect_face(frame)
            if box is not None:
                faces.append(crop_face(frame, box))
        idx += 1
    cap.release()
    if len(faces) < N_FRAMES:
        return None
    picks = np.linspace(0, len(faces) - 1, N_FRAMES).round().astype(int)
    return [faces[i] for i in picks]

def label_of(video_path):
    """Target specific subfolder names to avoid root directory name clashes."""
    p_lower = video_path.lower()
    if "dfd_original" in p_lower or "original sequences" in p_lower:
        return "real"
    elif "dfd_manipulated" in p_lower or "manipulated_sequences" in p_lower:
        return "fake"
    return None

videos = {"real": [], "fake": []}

print("Scanning /kaggle/input for videos...")
for dp, _, fs in os.walk("/kaggle/input"):
    for f in fs:
        if f.lower().endswith(VIDEO_EXT):
            p = os.path.join(dp, f)
            lbl = label_of(p)
            if lbl in videos:
                videos[lbl].append(p)

print(f"Found videos -> real: {len(videos['real'])} | fake: {len(videos['fake'])}")

if not videos["real"] or not videos["fake"]:
    raise SystemExit("Stopped: Missing one of the classes. Check directory paths.")


def run(label):
    vids = sorted(videos[label])
    random.shuffle(vids)
    vids = vids[:MAX_PER_CLASS]           # Limits the list so tqdm only tracks your target count
    
    done, tried = 0, 0
    for v in tqdm(vids, desc=label):
        if done >= MAX_PER_CLASS:               
            break
        tried += 1
        name = os.path.splitext(os.path.basename(v))[0]
        folder = os.path.join(OUT_DIR, label, name)
        if os.path.isdir(folder) and len(os.listdir(folder)) == N_FRAMES:
            done += 1                            
            continue
        faces = extract_faces(v)
        if faces is None:
            continue
        os.makedirs(folder, exist_ok=True)
        for i, face in enumerate(faces):
            cv2.imwrite(os.path.join(folder, f"{i:02d}.jpg"), face)
        done += 1
    print(f"{label}: {done} videos saved (checked {tried})")
    return done

n_real, n_fake = run("real"), run("fake")
print(f"\nDONE -> real: {n_real} videos, fake: {n_fake} videos "
      f"({n_real * N_FRAMES} + {n_fake * N_FRAMES} images) in {OUT_DIR}")

In [ ]:
import os
import glob
import random
from collections import defaultdict
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
from tqdm import tqdm


DATA_DIR = "/kaggle/working/frames"
EXPORT_DIR = "/kaggle/working/project_exports"
os.makedirs(EXPORT_DIR, exist_ok=True)

MODEL_NAME = "deepfake_detector_pro.pth"
MODEL_SAVE_PATH = os.path.join(EXPORT_DIR, MODEL_NAME)

BATCH_SIZE = 32
EPOCHS = 15
LEARNING_RATE = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
print(f"All outputs will be saved to: {EXPORT_DIR}")

class DeepfakeDataset(Dataset):
    def __init__(self, video_folders, transform=None):
        self.samples = []
        for folder in video_folders:
            label = 1 if "fake" in folder.lower() else 0
            frames = sorted(glob.glob(os.path.join(folder, "*.jpg")))
            for frame_path in frames:
                self.samples.append((frame_path, label, folder))
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label, folder = self.samples[idx]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long), folder

#data aug
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

#splitting
real_videos = glob.glob(os.path.join(DATA_DIR, "real", "*"))
fake_videos = glob.glob(os.path.join(DATA_DIR, "fake", "*"))

train_real, val_real = train_test_split(real_videos, test_size=0.2, random_state=42)
train_fake, val_fake = train_test_split(fake_videos, test_size=0.2, random_state=42)

train_dataset = DeepfakeDataset(train_real + train_fake, transform=train_transform)
val_dataset = DeepfakeDataset(val_real + val_fake, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
in_features = model.classifier[1].in_features

model.classifier = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(in_features, 256),
    nn.BatchNorm1d(256),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(256, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Dropout(p=0.2),
    nn.Linear(128, 2)
)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-2)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

history = {
    "train_loss": [], "val_loss": [],
    "train_acc": [], "val_frame_acc": [], "val_video_acc": []
}

best_video_acc = 0.0
best_video_probs = []
best_video_Targets = []

for epoch in range(EPOCHS):
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    
    for images, labels, _ in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]"):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)
        
    scheduler.step()
    
    epoch_train_loss = train_loss / train_total
    epoch_train_acc = train_correct / train_total

    # Validation Phase
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    video_preds = defaultdict(list)
    video_true_labels = {}

    with torch.no_grad():
        for images, labels, folders in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
            
            probs = torch.softmax(outputs, dim=1)[:, 1]
            for folder, prob, label in zip(folders, probs, labels):
                video_preds[folder].append(prob.item())
                video_true_labels[folder] = label.item()

    epoch_val_loss = val_loss / val_total
    epoch_val_acc = val_correct / val_total

    # Calculate Video-Level Metrics
    video_correct = 0
    current_probs, current_targets = [], []
    for folder, probs in video_preds.items():
        avg_prob = sum(probs) / len(probs)
        true_lbl = video_true_labels[folder]
        current_probs.append(avg_prob)
        current_targets.append(true_lbl)
        if (1 if avg_prob > 0.5 else 0) == true_lbl:
            video_correct += 1
            
    epoch_video_acc = video_correct / len(video_preds)

    history["train_loss"].append(epoch_train_loss)
    history["val_loss"].append(epoch_val_loss)
    history["train_acc"].append(epoch_train_acc)
    history["val_frame_acc"].append(epoch_val_acc)
    history["val_video_acc"].append(epoch_video_acc)

    print(f"Epoch {epoch+1}: "
          f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f} || "
          f"Val Loss: {epoch_val_loss:.4f} | Frame Acc: {epoch_val_acc:.4f} | **Video Acc: {epoch_video_acc:.4f}**")

   
    if epoch_video_acc >= best_video_acc:
        best_video_acc = epoch_video_acc
        best_video_probs = current_probs
        best_video_Targets = current_targets
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"--> Saved Best Model to {MODEL_SAVE_PATH} (Video Acc: {best_video_acc:.4f})")


epochs_range = range(1, EPOCHS + 1)


plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, history["train_acc"], label="Train Acc (Frame)", marker='o')
plt.plot(epochs_range, history["val_frame_acc"], label="Val Acc (Frame)", marker='s')
plt.plot(epochs_range, history["val_video_acc"], label="Val Acc (Video-Level)", marker='^', linewidth=2.5)
plt.title("Model Accuracy Curves")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(epochs_range, history["train_loss"], label="Train Loss", marker='o')
plt.plot(epochs_range, history["val_loss"], label="Val Loss", marker='s')
plt.title("Model Loss Curves")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

plt.tight_layout()
curves_path = os.path.join(EXPORT_DIR, "training_curves.png")
plt.savefig(curves_path, dpi=300)
plt.show()

# Plot 2: Video-Level ROC Curve
fpr, tpr, _ = roc_curve(best_video_Targets, best_video_probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Video ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Video-Level Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.grid(True)

roc_path = os.path.join(EXPORT_DIR, "video_roc_curve.png")
plt.savefig(roc_path, dpi=300)
plt.show()

print(f"\nAll done! Files saved inside {EXPORT_DIR}:")
print(os.listdir(EXPORT_DIR))

In [ ]:
import shutil
from IPython.display import FileLink

# Zip the entire project_exports folder
shutil.make_archive("/kaggle/working/deepfake_project_results", 'zip', "/kaggle/working/project_exports")

# Generate a clickable download link right below the cell
FileLink(r'deepfake_project_results.zip')